# Classification de déchets avec PySpark MLlib

Ce notebook montre comment entraîner un modèle de Machine Learning pour identifier le type de déchet à partir de vos images préalablement traitées par `data_processing.py`.

In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, array_position, array_max, flatten, transform
from pyspark.sql.types import StructType, StructField, IntegerType, ArrayType, DoubleType
from pyspark.ml.functions import array_to_vector, vector_to_array
from datetime import datetime

spark = SparkSession.builder \
   .appName("GarbageClassification") \
   .master("local[*]") \
   .getOrCreate()
sc = spark.sparkContext
print("----------------> "+sc.version)

I0000 00:00:1774361959.706543  103568 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774361959.707102  103568 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774361959.762495  103568 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774361961.037313  103568 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

I0000 00:00:1774361959.706543  103568 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774361959.707102  103568 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774361959.762495  103568 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774361961.037313  103568 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

----------------> 3.5.5


### Chargement et formatage des données
MLlib nécessite que les features soient sous forme de `Vector`. Nos pixels sont stockés au format texte dans le CSV, séparés par des virgules.

In [2]:
def load_and_prepare_data(path, split):
    df = spark.read.parquet(path)
    label_col = f"y_{split}"
    feature_col = f"x_{split}"

    df = df.withColumn(
        "label", 
        expr(f"array_position({label_col}, array_max({label_col})) - 1").cast("double")
    )

    df = df.withColumn(
        "features", 
        array_to_vector(
            transform(flatten(col(feature_col)), lambda x: x.cast("double"))
        )
    )

    return df.select("label", "features").dropna()

# Test
train_data = load_and_prepare_data("../data/train_data.parquet", "train")
test_data = load_and_prepare_data("../data/test_data.parquet", "test")
train_data.show(5)
test_data.show(5)

+-----+--------------------+
|label|            features|
+-----+--------------------+
|  4.0|[185.0,211.0,210....|
|  4.0|[175.0,198.0,196....|
|  4.0|[129.0,143.0,142....|
|  4.0|[190.0,217.0,214....|
|  4.0|[136.0,149.0,146....|
+-----+--------------------+
only showing top 5 rows

+-----+--------------------+
|label|            features|
+-----+--------------------+
|  0.0|[11.0,10.0,10.0,9...|
|  0.0|[254.0,254.0,254....|
|  0.0|[86.0,86.0,88.0,9...|
|  0.0|[106.0,111.0,113....|
|  0.0|[163.0,163.0,164....|
+-----+--------------------+
only showing top 5 rows



In [3]:
CONFIG = {
    "img_size": 64,
    "batch_size": 64,
    "epochs": 60,
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.05,
    "seed": 42
}

def data_augmentation():
    data_aug = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.04),
        tf.keras.layers.RandomZoom(0.05)
    ], name="data_augmentation")
    return data_aug

def create_cnn(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        data_augmentation(),
        tf.keras.layers.Rescaling(1.0 / 255.0),
        tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.20),
        tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.40),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ], name="CNN")
    
    return compile_model(model)


def compile_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG["learning_rate"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def spark_to_numpy(df):
    local_df = df.select(
        col("label").cast("int").alias("label"),
        vector_to_array("features").alias("features")
    )
    pdf = local_df.toPandas()
    X = np.vstack(pdf["features"].values).astype("float32")
    y = pdf["label"].to_numpy(dtype=np.int32)
    return X, y

def prepare_and_reshape(X, n_features):
    side = int(np.sqrt(n_features))
    if side * side == n_features:
        return X.reshape(-1, side, side, 1), (side, side, 1)
    return X[..., np.newaxis], (n_features, 1)

def save_model(model, name):
    filename = f"{name}.keras"
    save_path = os.path.join("..", "models", filename)
    model.save(save_path)
    print(f"Modèle sauvegardé : {save_path}")


def main():
    tf.keras.utils.set_random_seed(CONFIG["seed"])

    train_prepared = train_data.cache()

    train_split_df, val_df = train_prepared.randomSplit([0.9, 0.1], seed=CONFIG["seed"])

    X_train, y_train = spark_to_numpy(train_split_df)
    X_val, y_val = spark_to_numpy(val_df)

    n_features = X_train.shape[1]
    num_classes = int(max(y_train.max(), y_val.max()) + 1)

    X_train, input_shape = prepare_and_reshape(X_train, n_features)
    X_val, _ = prepare_and_reshape(X_val, n_features)
    print(f"Train: {X_train.shape}, Val: {X_val.shape}")
    print(f"Distribution val labels: {np.bincount(y_val)}")

    class_counts = np.bincount(y_train, minlength=num_classes)
    class_weight = {
        i: float(len(y_train) / (num_classes * count))
        for i, count in enumerate(class_counts) if count > 0
    }

    model = create_cnn(input_shape, num_classes)

    model.summary()

    #Entraînement
    log_name = f"{model.name}_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    current_log_path = os.path.join("..", "models", "logs", log_name)

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=CONFIG["epochs"],
        batch_size=CONFIG["batch_size"],
        class_weight=class_weight,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_accuracy", patience=8, restore_best_weights=True, mode="max"
            ),
            tf.keras.callbacks.TensorBoard(log_dir=current_log_path)
        ],
        verbose=2
    )
    save_model(model, f"final_{model.name}")
    spark.catalog.clearCache()


main()


26/03/24 15:19:36 WARN BlockManager: Putting block rdd_22_15 failed due to exception org.apache.spark.SparkException: Encountered error while reading file file:///home/mathis/Desktop/Spark/Garbage-Classification/data/train_data.parquet/part-00014-d973f1d4-6f5b-4e47-b806-338f4500f166-c000.snappy.parquet. Details:.
26/03/24 15:19:36 WARN BlockManager: Block rdd_22_15 could not be removed as it was not found on disk or in memory
26/03/24 15:19:36 ERROR Executor: Exception in task 15.0 in stage 4.0 (TID 19)
org.apache.spark.SparkException: Encountered error while reading file file:///home/mathis/Desktop/Spark/Garbage-Classification/data/train_data.parquet/part-00014-d973f1d4-6f5b-4e47-b806-338f4500f166-c000.snappy.parquet. Details:
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:864)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:296)
	at org.apache.spark.sql.execution.datasources.FileS

26/03/24 15:19:36 WARN BlockManager: Putting block rdd_22_15 failed due to exception org.apache.spark.SparkException: Encountered error while reading file file:///home/mathis/Desktop/Spark/Garbage-Classification/data/train_data.parquet/part-00014-d973f1d4-6f5b-4e47-b806-338f4500f166-c000.snappy.parquet. Details:.
26/03/24 15:19:36 WARN BlockManager: Block rdd_22_15 could not be removed as it was not found on disk or in memory
26/03/24 15:19:36 ERROR Executor: Exception in task 15.0 in stage 4.0 (TID 19)
org.apache.spark.SparkException: Encountered error while reading file file:///home/mathis/Desktop/Spark/Garbage-Classification/data/train_data.parquet/part-00014-d973f1d4-6f5b-4e47-b806-338f4500f166-c000.snappy.parquet. Details:
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:864)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:296)
	at org.apache.spark.sql.execution.datasources.FileS

ConnectionRefusedError: [Errno 111] Connection refused